# routes/selection

> Route handlers for selection queue management (remove, reorder, clear)

In [ ]:
#| default_exp routes.selection

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Any, Dict, Tuple, Callable
from pathlib import Path

from fasthtml.common import APIRouter

from cjm_fasthtml_file_browser.core.config import FileBrowserConfig
from cjm_fasthtml_file_browser.core.models import BrowserState
from cjm_fasthtml_file_browser.providers.local import LocalFileSystemProvider
from cjm_fasthtml_file_browser.routes.handlers import FileBrowserRouters
from cjm_workflow_state.state_store import SQLiteWorkflowStateStore

from cjm_transcription_source_select.models import SourceSelectUrls
from cjm_transcription_source_select.routes.core import (
    get_step_state, update_step_state, get_session_id_from_sess
)
from cjm_transcription_source_select.components.selection_panel import render_selection_panel
from cjm_transcription_source_select.components.stats_panel import render_stats_panel
from cjm_transcription_source_select.components.file_browser_panel import (
    get_browser_state, sync_browser_selection, render_browser_panel
)

DEBUG_REORDER = False

## OOB Browser Panel Helper

When selection changes via remove/clear, re-render the browser panel with updated selection highlights.

In [ ]:
#| export
def _render_oob_browser(
    fb_routers: FileBrowserRouters,  # File browser routers (has render + selection refs)
    selected_files: list,  # Current selected files
    selected_folders: list = None,  # Current selected folders
) -> Any:  # Browser panel with OOB attribute set
    """Render browser panel as OOB swap to update selection highlights."""
    # Sync selection into file browser's closure state
    sync_browser_selection(fb_routers._fb_state_getter(), selected_files, selected_folders)
    browser = render_browser_panel(render_fn=fb_routers.render)
    browser.attrs["hx-swap-oob"] = "outerHTML"
    return browser

## Remove Handler

In [ ]:
#| export
def _handle_remove(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    workflow_id: str,  # Workflow identifier
    urls: SourceSelectUrls,  # URL bundle
    fb_routers: FileBrowserRouters,  # File browser routers
    sess,  # FastHTML session
    path: str,  # File path to remove
):  # (selection panel, OOB browser panel, OOB stats panel)
    """Remove a file from the selection."""
    session_id = get_session_id_from_sess(sess)
    step_state = get_step_state(state_store, workflow_id, session_id)
    selected_files = step_state.get("selected_files", [])
    selected_folders = step_state.get("selected_folders", [])
    extraction_results = step_state.get("extraction_results", {})

    selected_files = [f for f in selected_files if f["path"] != path]
    extraction_results.pop(path, None)

    # Auto-deselect parent folder if no files from it remain
    parent = str(Path(path).parent)
    if parent in selected_folders:
        remaining = {f["path"] for f in selected_files}
        has_sibling = any(str(Path(p).parent) == parent for p in remaining)
        if not has_sibling:
            selected_folders = [f for f in selected_folders if f != parent]

    update_step_state(
        state_store, workflow_id, session_id,
        selected_files=selected_files,
        selected_folders=selected_folders,
        extraction_results=extraction_results,
        verified=False,
        committed_audio_paths=[],
    )

    # Sync browser selection display
    sync_browser_selection(fb_routers._fb_state_getter(), selected_files, selected_folders)

    selection = render_selection_panel(selected_files, urls, extraction_results)
    browser_oob = _render_oob_browser(fb_routers, selected_files, selected_folders)
    stats_oob = render_stats_panel(selected_files, urls, extraction_results, verified=False)
    stats_oob.attrs["hx-swap-oob"] = "outerHTML"
    return selection, browser_oob, stats_oob

## Reorder Handler

## Clear Handler

In [ ]:
#| export
async def _handle_reorder(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    workflow_id: str,  # Workflow identifier
    urls: SourceSelectUrls,  # URL bundle
    request,  # FastHTML request
    sess,  # FastHTML session
):  # Rendered selection panel
    """Reorder selected files based on SortableJS drag result."""
    session_id = get_session_id_from_sess(sess)
    step_state = get_step_state(state_store, workflow_id, session_id)
    selected_files = step_state.get("selected_files", [])

    # Extract new order from form data (Hidden inputs sent in DOM order)
    form_data = await request.form()
    new_order_paths = form_data.getlist("item")

    if DEBUG_REORDER:
        print(f"\n[REORDER DEBUG] Handler called")
        print(f"  Content-Type: {request.headers.get('content-type', 'MISSING')}")
        print(f"  form_data keys: {list(form_data.keys())}")
        print(f"  form_data.getlist('item'): {new_order_paths}")
        print(f"  len(new_order_paths): {len(new_order_paths)}")
        print(f"  len(selected_files): {len(selected_files)}")

    # Rebuild list in new order
    path_to_file = {f["path"]: f for f in selected_files}
    reordered = [path_to_file[p] for p in new_order_paths if p in path_to_file]

    # Add any files not in the form data (shouldn't happen, but safe)
    reordered_paths = {f["path"] for f in reordered}
    for f in selected_files:
        if f["path"] not in reordered_paths:
            reordered.append(f)

    update_step_state(
        state_store, workflow_id, session_id,
        selected_files=reordered,
    )

    return render_selection_panel(reordered, urls)

In [ ]:
#| export
def _handle_clear(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    workflow_id: str,  # Workflow identifier
    urls: SourceSelectUrls,  # URL bundle
    fb_routers: FileBrowserRouters,  # File browser routers
    sess,  # FastHTML session
):  # (selection panel, OOB browser panel, OOB stats panel)
    """Clear all selected files and folders."""
    session_id = get_session_id_from_sess(sess)

    update_step_state(
        state_store, workflow_id, session_id,
        selected_files=[],
        selected_folders=[],
        extraction_results={},
        verified=False,
        committed_audio_paths=[],
    )

    # Sync browser selection display
    sync_browser_selection(fb_routers._fb_state_getter(), [], [])

    selection = render_selection_panel([], urls)
    browser_oob = _render_oob_browser(fb_routers, [], [])
    stats_oob = render_stats_panel([], urls)
    stats_oob.attrs["hx-swap-oob"] = "outerHTML"
    return selection, browser_oob, stats_oob

## Router Initialization

In [ ]:
#| export
def init_selection_router(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    provider: LocalFileSystemProvider,  # File system provider (for OOB browser updates)
    config: FileBrowserConfig,  # Browser configuration (for OOB browser updates)
    workflow_id: str,  # Workflow identifier
    urls: SourceSelectUrls,  # Mutable URL bundle
    home_path: str = "",  # Home directory path
    fb_routers: FileBrowserRouters = None,  # File browser routers (for OOB sync)
    prefix: str = "/selection",  # Route prefix
) -> Tuple[APIRouter, Dict[str, Callable]]:  # (router, route_dict)
    """Initialize selection queue management routes."""
    router = APIRouter(prefix=prefix)

    @router
    def remove(sess, path: str = ""):
        """Remove a file from the selection."""
        if not path:
            session_id = get_session_id_from_sess(sess)
            step_state = get_step_state(state_store, workflow_id, session_id)
            return render_selection_panel(step_state.get("selected_files", []), urls)
        return _handle_remove(state_store, workflow_id, urls, fb_routers, sess, path)

    @router
    async def reorder(request, sess):
        """Reorder selected files via SortableJS."""
        return await _handle_reorder(state_store, workflow_id, urls, request, sess)

    @router
    def clear(sess):
        """Clear all selected files."""
        return _handle_clear(state_store, workflow_id, urls, fb_routers, sess)

    routes = {
        "remove": remove,
        "reorder": reorder,
        "clear": clear,
    }

    return router, routes

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()